# SpendGPT: make `user_id` required

Every `transactions`, `budgets` and `alerts` row must belong to a valid user (`user_id` can never be NULL).
This notebook alters the existing TiDB tables in place, so nothing needs to be recreated.

Steps: connect, check for NULL rows, apply `NOT NULL`, verify, test.

Needs a `.env` file in the same folder as this notebook with `TIDB_HOST`, `TIDB_PORT`, `TIDB_USER`, `TIDB_PASSWORD`, `TIDB_DATABASE`.

In [ ]:
import os
from pathlib import Path

import certifi
import mysql.connector
from dotenv import load_dotenv

env_path = Path.cwd() / '.env'
print('Looking for .env at:', env_path, '- exists:', env_path.exists())
load_dotenv(env_path)

conn = mysql.connector.connect(
    host=os.getenv('TIDB_HOST'),
    port=int(os.getenv('TIDB_PORT', 4000)),
    user=os.getenv('TIDB_USER'),
    password=os.getenv('TIDB_PASSWORD'),
    database=os.getenv('TIDB_DATABASE', 'student_spend_db'),
    ssl_ca=certifi.where(),
    ssl_verify_cert=True,
    ssl_verify_identity=True,
)
cur = conn.cursor()
print('Connected to TiDB.')

## 1. Check for existing NULL rows

`ALTER TABLE ... NOT NULL` fails if any row already has a NULL `user_id`.

In [ ]:
TABLES = ['transactions', 'budgets', 'alerts']

null_counts = {}
for table in TABLES:
    cur.execute(f'SELECT COUNT(*) FROM {table} WHERE user_id IS NULL')
    null_counts[table] = cur.fetchone()[0]

print('Rows with NULL user_id:', null_counts)
if any(null_counts.values()):
    print('Delete these rows or assign them to a real user before running the next cell.')
else:
    print('No NULL user_id rows. Safe to continue.')

## 2. Make `user_id` NOT NULL

Safe to re-run: setting an already-required column to `NOT NULL` changes nothing.

In [ ]:
if any(null_counts.values()):
    raise RuntimeError('NULL user_id rows still exist. Fix them first (see the previous cell).')

for table in TABLES:
    try:
        cur.execute(f'ALTER TABLE {table} MODIFY COLUMN user_id INT NOT NULL')
        print(f'{table}: user_id is now NOT NULL')
    except mysql.connector.Error as e:
        print(f'{table}: FAILED - {e}')

## 3. Verify

Every table that has a `user_id` column should show `nullable = NO`.

In [ ]:
cur.execute('''
    SELECT table_name, is_nullable
    FROM information_schema.columns
    WHERE table_schema = DATABASE() AND column_name = 'user_id'
    ORDER BY table_name
''')
for table_name, is_nullable in cur.fetchall():
    print(f'{table_name}: nullable = {is_nullable}')

## 4. Test: a budget with no user must be rejected

In [ ]:
cur.execute('SELECT category_id FROM categories LIMIT 1')
category_id = cur.fetchone()[0]

try:
    cur.execute(
        'INSERT INTO budgets (user_id, category_id, amount_limit, start_date, end_date) '
        'VALUES (NULL, %s, 10, %s, %s)',
        (category_id, '2026-11-01', '2026-11-30'),
    )
    conn.commit()
    print('PROBLEM: the insert with a NULL user_id succeeded!')
except mysql.connector.Error as e:
    conn.rollback()
    print('Good - the insert was rejected:', e)

In [ ]:
cur.close()
conn.close()
print('Connection closed.')